Final flow 
- all code in functions
- data pull from geoserver
- column descriptions and layer descriptions from csv
- style file from github
- output stored locally

1. add comments to functions
2. common functions between raster and vector 


1. fill in the functions
2. test for 1 layer

TODO: 
1. best id structure

In [29]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os

import json
import xml.etree.ElementTree as ET
import datetime
# from datetime import datetime, timezone

import urllib
import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon

import sys
sys.path.append('..')
import constants

import pystac
from pystac.extensions.table import TableExtension
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension

In [30]:
GEOSERVER_BASE_URL = constants.GEOSERVER_BASE_URL
GEOSERVER_BASE_URL

'https://geoserver.core-stack.org:8443/geoserver'

In [31]:
GITHUB_DATA_URL = constants.GITHUB_DATA_URL
GITHUB_DATA_URL

'https://raw.githubusercontent.com/Nirzaree/STAC-spec/stac-spec-common/data/'

In [32]:
RASTER_STYLE_PATH = '../data/LULC0_12class.qml'

In [33]:
LOCAL_DATA_DIR = '../data/'

In [34]:
STYLE_FILE_DIR = os.path.join(LOCAL_DATA_DIR,'input/style_files/')

In [35]:
THUMBNAIL_DIR = os.path.join(LOCAL_DATA_DIR,
                                     'STAC_output')
THUMBNAIL_DIR

'../data/STAC_output'

In [36]:
STAC_FILES_DIR = os.path.join(
    LOCAL_DATA_DIR,
    'CorestackCatalogs_test' #test folder
)
STAC_FILES_DIR

'../data/CorestackCatalogs_test'

Raster flow

In [37]:
def read_layer_description(filepath,
                           layer_name):
    layer_desc_df = pd.read_csv(filepath)
    layer_desc = layer_desc_df[layer_desc_df['layer_name'] == layer_name]['layer_description'].iloc[0]
    return layer_desc

In [38]:
def generate_raster_url(workspace,
                        layer_name,
                        geoserver_base_url,
                        output_format="geotiff"):
    wcs_url = (
        f"{geoserver_base_url}/{workspace}/wcs?"
        f"service=WCS&version=2.0.1&request=GetCoverage&"
        f"CoverageId={workspace}:{layer_name}&"
        f"format={output_format}"
    )
    print("Raster URL:",wcs_url)
    return wcs_url

In [39]:
def read_raster_data(raster_url):

    #when reading from geoserver
    response = requests.get(raster_url, verify=False)
    response.raise_for_status()
    raster_data = BytesIO(response.content)

    #read the data and fetch the metadata
    with rasterio.open(raster_data) as r:
        crs = r.crs
        bounds = r.bounds
        bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]
        footprint = Polygon([
            [bounds.left, bounds.bottom],
            [bounds.left, bounds.top],
            [bounds.right, bounds.top],
            [bounds.right, bounds.bottom]
        ])
        data = r.read(1) #TODO: wouldn't work if there are multiple bands

        # id = os.path.basename(raster_url) #works when data is local
        # id = layer_name
        gsd = 10
        shape = r.shape
        data_type = str(r.dtypes[0])
        
        return (data,
                bbox,
                mapping(footprint),
                crs,
                # id,
                gsd,
                shape,
                data_type
                )

In [40]:
# def compute_ground_sample_distance():

In [41]:
def create_raster_item(raster_filepath,
                       id,
                       layer_description
                       ):

    # raster_data,bbox,footprint,crs,id,gsd,shape,data_type = read_raster_data(raster_filepath)
    raster_data,bbox,footprint,crs,gsd,shape,data_type = read_raster_data(raster_filepath)

    raster_item = pystac.Item(id=id,
                        geometry=footprint,
                        bbox=bbox,
                        datetime=datetime.datetime.now(datetime.timezone.utc),
                        properties={
                            #   title
                            "description" : layer_description,
                            # "gsd": gsd, #adding this in raster extension 
                        })
    
    #add certain metadata under projection extension
    proj_ext = ProjectionExtension.ext(raster_item, add_if_missing=True)
    proj_ext.epsg = crs
    proj_ext.shape = [shape[0], shape[1]]

    return (raster_item,raster_data) #raster_data is needed for thumbnail generation so returning that as well

In [42]:
def add_raster_data_asset(raster_item,
                          geoserver_url
                          ):
    raster_item.add_asset("data", Asset(
    # href=os.path.join(data_url, os.path.relpath(raster_path, start=data_dir)), #TODO
    href=geoserver_url,
    roles=["data"],
    title="Raster Layer"))

    return raster_item

In [43]:
# def add_raster_extension(raster_item): #TODO
#         #add certain metadata under raster extension
#     raster_ext = RasterExtension.ext(raster_item.assets["data"], add_if_missing=True)
#     raster_band = RasterBand.create(
#         data_type=data_type, 
#         spatial_resolution=gsd,
#         # nodata=nodata
#     )
#     raster_ext.bands = [raster_band]  

In [44]:
def parse_raster_style_file(style_file_url,
                            STYLE_FILE_DIR
                            ):
    
    #download style file if not already downloaded, and save it locally
    style_file_name = os.path.basename(style_file_url)
    style_file_local_path = os.path.join(STYLE_FILE_DIR,
                                         style_file_name)
    
    if not style_file_local_path:
            #TODO: try statement
            urllib.request.urlretrieve(style_file_url,
                                       style_file_local_path)       
    
    tree = ET.parse(style_file_local_path)
    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        classes.append(class_info)

    # If no paletteEntry tags are found, check for item tags
    if not classes:
        for entry in root.findall(".//item"):
            class_info = {}
            for attr_key, attr_value in entry.attrib.items():
                if attr_key == "value":
                    try:
                        class_info[attr_key] = int(attr_value)
                    except ValueError:
                        class_info[attr_key] = attr_value
                else:
                    class_info[attr_key] = attr_value
            classes.append(class_info)
    return classes

In [45]:
def add_classification_extension(raster_style_url,
                                 raster_item
                                 ):
    
    style_info = parse_raster_style_file(style_file_url=raster_style_url,
                                         STYLE_FILE_DIR=STYLE_FILE_DIR
                                         )
    classification_ext = ClassificationExtension.ext(raster_item.assets["data"], add_if_missing=True)
    stac_classes = []
    for cls in style_info:
        stac_class_obj = Classification.create(
            value=int(cls["value"]),
            name=cls.get("label") or f"Class {cls['value']}",
            description=cls.get("label"),
            color_hint=cls['color'].replace('#','')
        )
        stac_classes.append(stac_class_obj)
    classification_ext.classes = stac_classes

    return (raster_item,style_info) #style info is required for thumbnail 

In [46]:
def add_stylefile_asset(raster_item,
                        style_file_url):
    raster_item.add_asset("style", Asset(
        # href=os.path.join(data_url, os.path.relpath(raster_style_path, start=data_dir)),
        href=style_file_url,
        media_type=MediaType.XML,
        roles=["metadata"],
        title="QGIS Style file"
    ))
    return raster_item

In [47]:
def generate_raster_thumbnail(raster_data,
                              style_info,
                              output_path
                              ):
    
    unique_raster_values = np.unique(raster_data.compressed() if isinstance(raster_data, np.ma.MaskedArray) else raster_data)
    # Filter QML info to only include values present in the raster data
    filtered_style_info = [cls for cls in style_info if cls.get('value') in unique_raster_values]
    
    values = [cls['value'] for cls in filtered_style_info if 'value' in cls]
    colors = [cls['color'] for cls in filtered_style_info if 'color' in cls]
    
    # print(f"Parsed QML values: {values}")
    # print(f"Parsed QML colors: {colors}")
        
    try:
        if not values or not colors or len(values) != len(colors):
            raise ValueError("Invalid or insufficient palette information in QML file.")
    
        sorted_indices = np.argsort(values)
        sorted_values = np.array(values)[sorted_indices]
        sorted_colors = np.array(colors)[sorted_indices]

        cmap = ListedColormap(sorted_colors)
        bounds = np.array(sorted_values) - 0.5
        bounds = np.append(bounds, sorted_values[-1] + 0.5)
        norm = Normalize(vmin=bounds.min(), vmax=bounds.max())

    except ValueError as e:
        print(f"Skipping palette generation due to error: {e}. Using a default colormap.")
        cmap = 'gray'
        norm = None
    plt.figure(figsize=(3, 3), dpi=100)
    
    plt.imshow(raster_data, cmap=cmap, norm=norm, interpolation='none')
    plt.axis('off')

    #os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close()

In [48]:
def add_thumbnail_asset(raster_item,
                        RASTER_THUMBNAIL_PATH,
                        LOCAL_DATA_DIR, #TODO
                        GITHUB_DATA_URL
                        ):
    raster_item.add_asset("thumbnail", Asset(
        href=os.path.join(GITHUB_DATA_URL, os.path.relpath(RASTER_THUMBNAIL_PATH, start=LOCAL_DATA_DIR)),
        media_type=MediaType.PNG,
        roles=["thumbnail"],
        title="Raster Thumbnail (QML)"
    ))

    return raster_item

In [49]:
def read_layer_mapping(layer_map_csv_path, #TODO: update this function for each type of layer
                       layer_name,
                       district, #
                       block,
                       start_year = '',
                       end_year = ''
                       ):
    layer_mapping_df = pd.read_csv(layer_map_csv_path)
    geoserver_workspace_name = layer_mapping_df[layer_mapping_df['layer_name'] == layer_name]['geoserver_workspace_name'].iloc[0]
    geoserver_layer_name = layer_mapping_df[layer_mapping_df['layer_name'] == layer_name]['geoserver_layer_name'].iloc[0]
    
    style_file_url = layer_mapping_df[layer_mapping_df['layer_name'] == layer_name]['style_file_url'].iloc[0]

    if (layer_name == 'land_use_land_cover_raster'):
        start_year = str(int(start_year) % 100) #keep only last 2 digits of the full year
        end_year = str(int(end_year) % 100)
        geoserver_layer_name = geoserver_layer_name.format(start_year = start_year,
                                                    end_year = end_year,
                                                    block = block)    
    # print(geoserver_workspace_name,geoserver_layer_name)
    elif ((layer_name == 'tree_canopy_cover_density_raster') |\
          (layer_name == 'tree_canopy_height_raster')):
                geoserver_layer_name = geoserver_layer_name.format(start_year = start_year,
                                                                   district = district,
                                                                   block = block)
    else: #only LULC has some specific things such as 2 digit year representation and no district information
        geoserver_layer_name = geoserver_layer_name.format(district = district,
                                                           block = block)
    return (geoserver_workspace_name,
            geoserver_layer_name,
            style_file_url
            )

In [50]:
def generate_raster_stac(state,
                         district,
                         block,
                         layer_name,
                         layer_map_csv_path,
                         layer_desc_csv_path,
                         start_year='',
                         end_year=''
                         ):    
    
    #1. read layer description
    layer_description = read_layer_description(filepath= layer_desc_csv_path,
                                               layer_name=layer_name)
    
    #2. get geoserver url parameters from the layer details
    geoserver_workspace_name,geoserver_layer_name,style_file_url = \
        read_layer_mapping(layer_map_csv_path = layer_map_csv_path,
                           district = district,
                           block=block,
                           layer_name=layer_name,
                           start_year=start_year,
                           end_year=end_year
                           )
    print(f"geoserver_workspace_name={geoserver_workspace_name}")
    print(f"geoserver_layer_name={geoserver_layer_name}")

    #3. generate geoserver url
    geoserver_url = generate_raster_url(workspace=geoserver_workspace_name,
                                        layer_name=geoserver_layer_name,
                                        geoserver_base_url=GEOSERVER_BASE_URL)
    
    #4. create raster item
    raster_item,raster_data = create_raster_item(geoserver_url,
                                                 id=geoserver_layer_name,
                                                 layer_description = layer_description
                                                 )
    
    #5. add raster data asset
    raster_item = add_raster_data_asset(raster_item,
                                        geoserver_url=geoserver_url)
    
    #6. add classification extension
    raster_item,style_info = add_classification_extension(raster_style_url=style_file_url,
                                                          raster_item=raster_item)
    
    #7. add style file asset
    add_stylefile_asset(raster_item=raster_item,
                        style_file_url=style_file_url
                        )
    
    #8. generate thumbnail
    if (start_year != ''):
        thumbnail_filename = f'{block}_{layer_name}_{start_year}.png'
    else:
        thumbnail_filename = f'{block}_{layer_name}.png' #TODO:
    RASTER_THUMBNAIL_PATH = os.path.join(THUMBNAIL_DIR,
                                         thumbnail_filename)
    
    generate_raster_thumbnail(raster_data=raster_data,
                              style_info=style_info,
                              output_path=RASTER_THUMBNAIL_PATH
                              )
    
    #9. add thumbnail asset
    raster_item = add_thumbnail_asset(
        raster_item=raster_item,
        RASTER_THUMBNAIL_PATH=RASTER_THUMBNAIL_PATH,
        LOCAL_DATA_DIR=LOCAL_DATA_DIR,
        GITHUB_DATA_URL=GITHUB_DATA_URL
    )

    return raster_item

In [51]:
def update_STAC_files(state,
                      district,
                      block,
                      STAC_item,
                      corestack_dir,#TODO
                    #   block_title,#TODO
                    #   district_title,#TODO
                    #   state_title #TODO
                      ):
    #1. create block catalog,if not already existing 
    block_dir = os.path.join(corestack_dir,
                             state,
                             district,
                             block)
    os.makedirs(block_dir, exist_ok=True)
    block_catalog_path = os.path.join(block_dir,'catalog.json')

    #if it already exists, read the catalog
    if os.path.exists(block_catalog_path):
        block_catalog = pystac.read_file(block_catalog_path)
        print(f"Loaded existing block catalog: {block}")

        #If block exists already then also district,state and root exist so nothing needs to be done further! 
        #just save the updated block 
         #2. add item to block catalog. 
        block_catalog.add_item(STAC_item)
        block_catalog.normalize_and_save(block_dir,
                                         catalog_type=pystac.CatalogType.SELF_CONTAINED)
        return 
    else:
        block_catalog = pystac.Catalog(
            id=block,
            title=f"STAC for {block}",
            description=f"STAC catalog for {block} block data in {district}, {state}")

        print("created block catalog")
        # block_catalog = pystac.Catalog(
        #     id=block,
        #     title=f"STAC for {block_title}",
        #     description=f"STAC catalog for {block_title} block data in {district_title}, {state_title}")

    #2. add item to block catalog. 
    block_catalog.add_item(STAC_item)
    
    #3. create district catalog if not existing
    district_dir = os.path.join(corestack_dir,
                             state,
                             district)
    os.makedirs(district_dir, exist_ok=True)
    district_catalog_path = os.path.join(district_dir,'catalog.json')

    if os.path.exists(district_catalog_path):
        district_catalog = pystac.read_file(district_catalog_path)
        print("loaded district catalog")
        #add block catalog to the district
        district_catalog.add_child(block_catalog)
        district_catalog.normalize_and_save(district_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
        return
    else: 
        district_catalog = pystac.Catalog(
            id=district,
            title= f"{district}",#f"{district_title}",
            description=f"STAC catalog for data of {district} district"
        )   
        print("created district catalog")
        #add block catalog to the district
        district_catalog.add_child(block_catalog)
        district_catalog.normalize_and_save(district_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)     

    #4. create state collection if not existing
    state_dir = os.path.join(corestack_dir,
                             state)
    state_collection_path = os.path.join(state_dir, "collection.json")

    if os.path.exists(state_collection_path):
        state_collection = pystac.read_file(state_collection_path)
        print("loaded state collection")
        state_collection.add_child(district_catalog)
        state_collection.normalize_and_save(state_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
        return 
    else:
        state_collection = pystac.Collection(
            id=state,
            title= state,#f"{state_title}",
            description=f"STAC Collection for data of {state} state.",
            extent=pystac.Extent(
                spatial=pystac.SpatialExtent([0, 0, 0, 0]), #TODO: let it come automatically from data, or make it India wide
                temporal=pystac.TemporalExtent([[constants.DEFAULT_START_DATE, 
                                                 constants.DEFAULT_END_DATE]])),
            license="https://spdx.org/licenses/CC-BY-4.0.html",
        providers=[
            pystac.Provider(
                name="CoRE Stack",
                roles=[pystac.ProviderRole.PRODUCER, pystac.ProviderRole.PROCESSOR, pystac.ProviderRole.HOST, pystac.ProviderRole.LICENSOR ],
                url="https://core-stack.org/"
            )
        ],
        keywords=["social-ecological", "sustainability","CoRE stack", block, district, state],
        )        #TODO: move this to constants and not hardcode it here. 
        print("created state collection")
        state_collection.add_child(district_catalog)
        state_collection.normalize_and_save(state_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
        
    #5. create root catalog if not existing
    root_catalog_path = os.path.join(corestack_dir, "catalog.json")
    if os.path.exists(root_catalog_path):
        root_catalog = pystac.read_file(root_catalog_path)
        print("loaded root catalog")
    else:
        os.makedirs(corestack_dir, exist_ok=True)
        root_catalog = pystac.Catalog(
            id="corestack_STAC",
            title=constants.ROOT_CATALOG_TITLE,
            description=constants.ROOT_CATALOG_DESCRIPTION
        )
        root_catalog.set_self_href(root_catalog_path)
        print("created root catalog")
    root_catalog.add_child(state_collection)
    root_catalog.normalize_and_save(corestack_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
    return

Test raster flow for a layer

In [52]:
block_district_state_df = pd.DataFrame({
    'block' : ['gobindpur','mirzapur','koraput','badlapur'],
    'district' : ['saraikela-kharsawan','mirzapur','koraput','jaunpur'],
    'state' : ['jharkhand','uttar_pradesh','odisha','uttar_pradesh']
})

block_district_state_df

,block,district,state
0,gobindpur,saraikela-kharsawan,jharkhand
1,mirzapur,mirzapur,uttar_pradesh
2,koraput,koraput,odisha
3,badlapur,jaunpur,uttar_pradesh


In [53]:
block = 'badlapur'
district = block_district_state_df[block_district_state_df['block'] == block]['district'].iloc[0]
state = block_district_state_df[block_district_state_df['block'] == block]['state'].iloc[0]
print(state,district,block)

uttar_pradesh jaunpur badlapur


In [ ]:
raster_item = generate_raster_stac(state=state,
                                   district=district,
                                   block=block,
                                   layer_name='land_use_land_cover_raster',
                                   layer_map_csv_path='../data/test_mapping.csv',
                                   layer_desc_csv_path='../data/layer_descriptions.csv',
                                   start_year='2022',
                                   end_year='2023')

geoserver_workspace_name=LULC_level_3
geoserver_layer_name=LULC_22_23_badlapur_level_3
Raster URL: https://geoserver.core-stack.org:8443/geoserver/LULC_level_3/wcs?service=WCS&version=2.0.1&request=GetCoverage&CoverageId=LULC_level_3:LULC_22_23_badlapur_level_3&format=geotiff


/home/nirzaree/miniconda3/envs/.stac/lib/python3.13/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'geoserver.core-stack.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
raster_item = generate_raster_stac(state=state,
                                   district=district,
                                   block=block,
                                   layer_name='terrain_raster',
                                   layer_map_csv_path='../data/test_mapping.csv',
                                   layer_desc_csv_path='../data/layer_descriptions.csv',
                                   )

In [ ]:
raster_item = generate_raster_stac(state=state,
                                   district=district,
                                   block=block,
                                   layer_name='tree_canopy_cover_density_raster',
                                   layer_map_csv_path='../data/test_mapping.csv',
                                   start_year='2022',
                                   end_year='2023',
                                   layer_desc_csv_path='../data/layer_descriptions.csv',
                                   )

In [ ]:
raster_item

In [ ]:
update_STAC_files(
    state=state,
    district=district,
    block=block,
    STAC_item=raster_item,
    corestack_dir=STAC_FILES_DIR
)